# Stage-2 n₀ refresh at 32 points (prompt 27, post-top-up)

`stage2_C_n0b` (rows 17–32 of the same Sobol sequence, collector commit
`127c8a4`) landed 09-22. This notebook re-runs the **pre-registered GP LOO
diagnostics with the T4 apparatus unchanged** (same Matérn-2.5 ARD kernel,
WhiteKernel, per-fold min-max scaling, same pinned noise floors from
pi_0911 §3.5) on all 32 points, against the frozen 16-point record in
`n0_gp_loo.csv`. The 16-point run found curtailment/cost **model-limited**
(7–17× floor) and shed already noise-limited (0.5×) — the top-up's job was
to pull the model-limited errors down.

**Caveat carried on every nuclear-influenced number:** g1 fuel-deletion
APPROX (patch pending PI review). Draft M from LOCATION.md v1 =
{shed, curt, cost_ls} — those three rows are the ones that gate BO round 1.

In [1]:
import json, os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.labelweight": "bold", "axes.titleweight": "bold",
                     "font.weight": "bold"})   # house style: bold axes

CAMPAIGN = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, CAMPAIGN)
from tiers import TIERS

OBJ_COLS = ["load_shed_mwh", "true_curtailment_mwh", "total_cost_raw_usd",
            "total_cost_less_synthetic_usd", "reserve_shortfall_mwh", "thermal_starts"]
FLOORS = {"load_shed_mwh": 3000.0, "true_curtailment_mwh": 5000.0,
          "total_cost_raw_usd": 0.5e6, "total_cost_less_synthetic_usd": 0.5e6,
          "reserve_shortfall_mwh": None, "thermal_starts": None}  # pi_0911 §3.5
DRAFT_M = ["load_shed_mwh", "true_curtailment_mwh", "total_cost_less_synthetic_usd"]
SORTED_TIERS = sorted(TIERS)

def load_wave(name):
    wdir = os.path.join(CAMPAIGN, "waves", name)
    dm = pd.read_csv(os.path.join(wdir, "design_matrix.csv"))
    ob = pd.read_csv(os.path.join(wdir, "objectives.csv"))
    return dm.merge(ob[["index"] + OBJ_COLS], on="index", validate="1:1")

n0 = load_wave("stage2_C_n0")
n0b = load_wave("stage2_C_n0b")
assert len(n0) == 16 and len(n0b) == 16
mb = json.load(open(os.path.join(CAMPAIGN, "waves", "stage2_C_n0b", "manifest.json")))["sobol"]
assert mb["skip"] == 16 and mb["n_drawn_total"] == 32
assert mb["continues_wave"] == "stage2_C_n0"
print("n0b sobol record:", {k: v for k, v in mb.items() if k != "lattice"})

both = pd.concat([n0.assign(wave="n0"), n0b.assign(wave="n0b")], ignore_index=True)
X = both[[f"{t}_omega" for t in SORTED_TIERS]].to_numpy(float)
assert not np.isnan(X).any() and len(np.unique(X, axis=0)) == 32
os.makedirs("figs", exist_ok=True)

n0b sobol record: {'seed': 20260821, 'skip': 16, 'n': 16, 'n_drawn_total': 32, 'continues_wave': 'stage2_C_n0', 'scipy_version': '1.16.3'}


## 1. GP LOO at 32 points vs the frozen 16-point record

`loo_preds` is byte-identical to `stage2_quicklook.ipynb` cell 9 (the
pre-registered apparatus) — only the training set changed 16 → 32.

In [2]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.exceptions import ConvergenceWarning

def loo_preds(X, y, seed=0):
    n = len(y)
    preds = np.empty(n)
    for i in range(n):
        tr = np.arange(n) != i
        Xtr, ytr = X[tr], y[tr]
        lo = Xtr.min(axis=0); span = np.where((Xtr.max(0) - lo) <= 0, 1.0, Xtr.max(0) - lo)
        kernel = (ConstantKernel(1.0, (1e-3, 1e3))
                  * Matern(length_scale=np.ones(X.shape[1]),
                           length_scale_bounds=(1e-2, 1e1), nu=2.5)
                  + WhiteKernel(1e-4, (1e-6, 1e0)))
        gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                       n_restarts_optimizer=1, random_state=seed,
                                       alpha=1e-8)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            gpr.fit((Xtr - lo) / span, ytr)
        preds[i] = gpr.predict(((X[i] - lo) / span).reshape(1, -1))[0]
    return preds

old = pd.read_csv("n0_gp_loo.csv").set_index("objective")  # frozen 16-pt record

rows, preds_by_obj = [], {}
for obj in OBJ_COLS:
    y = both[obj].to_numpy(float)
    p = loo_preds(X, y)
    preds_by_obj[obj] = p
    rmse = float(np.sqrt(np.mean((p - y) ** 2)))
    rng = float(y.max() - y.min())
    row = {"objective": obj, "loo_rmse_32": rmse, "y_range_32": rng,
           "rmse_over_range_32": rmse / rng,
           "loo_R2_32": float(1 - np.sum((p - y) ** 2) / np.sum((y - y.mean()) ** 2)),
           "loo_rmse_16": float(old.loc[obj, "loo_rmse"]),
           "loo_R2_16": float(old.loc[obj, "loo_R2"]),
           "rmse_ratio_32_over_16": rmse / float(old.loc[obj, "loo_rmse"])}
    if FLOORS[obj]:
        row["rmse_over_floor_32"] = rmse / FLOORS[obj]
        row["rmse_over_floor_16"] = float(old.loc[obj, "rmse_over_floor"])
    rows.append(row)
loo32 = pd.DataFrame(rows)
loo32.to_csv("n0_32pt_gp_loo.csv", index=False)
with pd.option_context("display.width", 200):
    print(loo32.round(3).to_string(index=False))

                    objective  loo_rmse_32   y_range_32  rmse_over_range_32  loo_R2_32  loo_rmse_16  loo_R2_16  rmse_ratio_32_over_16  rmse_over_floor_32  rmse_over_floor_16
                load_shed_mwh     1206.847 1.372931e+04               0.088      0.894     1497.157      0.876                  0.806               0.402               0.499
         true_curtailment_mwh    33116.151 5.400735e+05               0.061      0.940    85361.692      0.675                  0.388               6.623              17.072
           total_cost_raw_usd  1440541.260 1.010042e+08               0.014      0.997  3619112.550      0.984                  0.398               2.881               7.238
total_cost_less_synthetic_usd  2139606.881 1.025306e+08               0.021      0.994  3557638.744      0.985                  0.601               4.279               7.115
        reserve_shortfall_mwh     5246.623 5.464611e+04               0.096      0.887     7011.109      0.831                  0.

In [3]:
floored = [o for o in OBJ_COLS if FLOORS[o]]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
x = np.arange(len(floored))
f16 = [loo32.set_index("objective").loc[o, "rmse_over_floor_16"] for o in floored]
f32 = [loo32.set_index("objective").loc[o, "rmse_over_floor_32"] for o in floored]
ax.bar(x - 0.18, f16, width=0.36, label="16 points", color="tab:gray")
ax.bar(x + 0.18, f32, width=0.36, label="32 points", color="tab:blue")
ax.axhline(1.0, color="crimson", lw=1.5, ls="--", label="noise floor")
ax.set_xticks(x); ax.set_xticklabels([o.replace("_", "\n") for o in floored], fontsize=8)
ax.set_ylabel("LOO RMSE / noise floor"); ax.set_yscale("log")
ax.set_title("Model-limited → noise-limited progress")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

ax = axes[1]
r16 = [loo32.set_index("objective").loc[o, "loo_R2_16"] for o in OBJ_COLS]
r32 = [loo32.set_index("objective").loc[o, "loo_R2_32"] for o in OBJ_COLS]
x = np.arange(len(OBJ_COLS))
ax.bar(x - 0.18, r16, width=0.36, label="16 points", color="tab:gray")
ax.bar(x + 0.18, r32, width=0.36, label="32 points", color="tab:blue")
ax.set_xticks(x); ax.set_xticklabels([o.replace("_", "\n") for o in OBJ_COLS], fontsize=7)
ax.set_ylabel("LOO R²"); ax.set_title("Surrogate skill by objective")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
fig.tight_layout(); fig.savefig("figs/loo_16_vs_32.png", dpi=150); plt.close(fig)
print("figs/loo_16_vs_32.png written")

figs/loo_16_vs_32.png written


In [4]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, obj in zip(axes.flat, OBJ_COLS):
    y = both[obj].to_numpy(float); p = preds_by_obj[obj]
    is_b = (both["wave"] == "n0b").to_numpy()
    ax.scatter(y[~is_b], p[~is_b], color="tab:gray", label="n0 rows", s=28)
    ax.scatter(y[is_b], p[is_b], color="tab:blue", label="n0b rows", s=28)
    lims = [min(y.min(), p.min()), max(y.max(), p.max())]
    ax.plot(lims, lims, "k:", lw=1)
    if FLOORS[obj]:
        ax.set_title(f"{obj}\nRMSE/floor {np.sqrt(np.mean((p-y)**2))/FLOORS[obj]:.1f}x", fontsize=9)
    else:
        ax.set_title(obj, fontsize=9)
    ax.set_xlabel("actual", fontsize=8); ax.set_ylabel("LOO prediction", fontsize=8)
axes.flat[0].legend(fontsize=8)
fig.suptitle("LOO predicted vs actual, 32-point n0 (g1 APPROX caveat on nuclear-heavy rows)",
             fontweight="bold")
fig.tight_layout(); fig.savefig("figs/loo_scatter_32.png", dpi=150); plt.close(fig)
print("figs/loo_scatter_32.png written")

figs/loo_scatter_32.png written


## 2. Objective ranges: did n0b extend coverage?

In [5]:
hull = load_wave("contour_303x317_C")
range_rows = []
for obj in OBJ_COLS:
    range_rows.append({
        "objective": obj,
        "n0_min": float(n0[obj].min()), "n0_max": float(n0[obj].max()),
        "n0b_min": float(n0b[obj].min()), "n0b_max": float(n0b[obj].max()),
        "combined_min": float(both[obj].min()), "combined_max": float(both[obj].max()),
        "hull_min": float(hull[obj].min()), "hull_max": float(hull[obj].max()),
        "n0b_extends_low": bool(n0b[obj].min() < n0[obj].min()),
        "n0b_extends_high": bool(n0b[obj].max() > n0[obj].max()),
    })
ranges32 = pd.DataFrame(range_rows)
ranges32.to_csv("n0_32pt_ranges.csv", index=False)
print(ranges32.round(1).to_string(index=False))

                    objective      n0_min      n0_max     n0b_min     n0b_max  combined_min  combined_max    hull_min    hull_max  n0b_extends_low  n0b_extends_high
                load_shed_mwh        22.4     13729.3         0.0      9986.5           0.0       13729.3      2920.2     40102.5             True             False
         true_curtailment_mwh     76268.9    616342.4     89901.1    547862.6       76268.9      616342.4    470809.1   1494291.7            False             False
           total_cost_raw_usd 546956562.0 647870883.6 560788064.9 647960717.6   546956562.0   647960717.6 528327810.7 578550575.0            False              True
total_cost_less_synthetic_usd 539251294.1 641212849.2 553368561.9 641781868.7   539251294.1   641781868.7 526484930.8 572048842.7            False              True
        reserve_shortfall_mwh      7721.6     59038.1      4392.0     57026.4        4392.0       59038.1     43220.9    190437.4             True             False
          

In [6]:
li = loo32.set_index("objective")
verdict = []
for obj in OBJ_COLS:
    if not FLOORS[obj]:
        continue
    f16, f32 = li.loc[obj, "rmse_over_floor_16"], li.loc[obj, "rmse_over_floor_32"]
    tag = " [draft-M]" if obj in DRAFT_M else ""
    if f32 <= 1.5:
        state = "NOISE-LIMITED — more n0 points buy ~nothing for this objective"
    elif f32 < f16 / 2:
        state = "still model-limited but improving fast (halved or better)"
    elif f32 < f16:
        state = "still model-limited; improvement modest"
    else:
        state = "NOT improved by the top-up — flag for review"
    verdict.append(f"{obj}{tag}: {f16:.1f}x -> {f32:.1f}x floor; {state}")
print("\n".join(verdict))
print()
m_floored = [o for o in DRAFT_M if FLOORS[o]]
worst = max(float(li.loc[o, "rmse_over_floor_32"]) for o in m_floored)
print(f"worst draft-M rmse/floor at 32 points: {worst:.1f}x")
print("Reminder: BO round 1 still waits on the FINAL M (prompt 28); the 32->64 "
      "question was NOT pre-registered — these numbers inform it, the decision is Kay/PI's.")

summary = {
    "collector_commit": "127c8a4",
    "n_points": 32,
    "gp_loo_32": loo32.to_dict("records"),
    "ranges_32": ranges32.to_dict("records"),
    "verdict_lines": verdict,
}
with open("t4b_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nt4b_summary.json written")

load_shed_mwh [draft-M]: 0.5x -> 0.4x floor; NOISE-LIMITED — more n0 points buy ~nothing for this objective
true_curtailment_mwh [draft-M]: 17.1x -> 6.6x floor; still model-limited but improving fast (halved or better)
total_cost_raw_usd: 7.2x -> 2.9x floor; still model-limited but improving fast (halved or better)
total_cost_less_synthetic_usd [draft-M]: 7.1x -> 4.3x floor; still model-limited; improvement modest

worst draft-M rmse/floor at 32 points: 6.6x
Reminder: BO round 1 still waits on the FINAL M (prompt 28); the 32->64 question was NOT pre-registered — these numbers inform it, the decision is Kay/PI's.

t4b_summary.json written
